# **Sentiment Analysis – Twitter Dataset**
## *Logistic Regression · Naive Bayes · TextBlob*
---

## **1. Carga de Datos**
Se cargan los dos datasets provistos para entrenamiento y evaluación.

In [11]:

import pandas as pd

train_path = '/home/Fede/training.1600000.processed.noemoticon.csv'
test_path = '/home/Fede/testdata.manual.2009.06.14.csv'

train_df = pd.read_csv(train_path, encoding='latin-1', header=None)
test_df = pd.read_csv(test_path, encoding='latin-1', header=None)

train_df.columns = ['polarity','id','date','query','user','text']
test_df.columns = ['polarity','id','date','query','user','text']

train_df.head(), test_df.head()


(   polarity          id                          date     query  \
 0         0  1467810369  Mon Apr 06 22:19:45 PDT 2009  NO_QUERY   
 1         0  1467810672  Mon Apr 06 22:19:49 PDT 2009  NO_QUERY   
 2         0  1467810917  Mon Apr 06 22:19:53 PDT 2009  NO_QUERY   
 3         0  1467811184  Mon Apr 06 22:19:57 PDT 2009  NO_QUERY   
 4         0  1467811193  Mon Apr 06 22:19:57 PDT 2009  NO_QUERY   
 
               user                                               text  
 0  _TheSpecialOne_  @switchfoot http://twitpic.com/2y1zl - Awww, t...  
 1    scotthamilton  is upset that he can't update his Facebook by ...  
 2         mattycus  @Kenichan I dived many times for the ball. Man...  
 3          ElleCTF    my whole body feels itchy and like its on fire   
 4           Karoli  @nationwideclass no, it's not behaving at all....  ,
    polarity  id                          date    query      user  \
 0         4   3  Mon May 11 03:17:40 UTC 2009  kindle2    tpryan   
 1         4 

## **2. Preprocesamiento**
Incluye normalización básica y lematización con spaCy (modelo inglés).

In [ ]:

import re
import spacy
nlp = spacy.load('en_core_web_sm')

def preprocess(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    doc = nlp(text)
    tokens = [token.lemma_ for token in doc if not token.is_stop]
    return ' '.join(tokens)

train_df['clean'] = train_df['text'].apply(preprocess)
test_df['clean'] = test_df['text'].apply(preprocess)

train_df[['text','clean']].head()


## **3. Vectorización (TF‑IDF)**
Se construyen vectores con n‑gramas (1–2).

In [ ]:

from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(ngram_range=(1,2), min_df=5)
X_train = vectorizer.fit_transform(train_df['clean'])
X_test = vectorizer.transform(test_df['clean'])

y_train = train_df['polarity']
y_test = test_df['polarity']

X_train.shape, X_test.shape


## **4. Entrenamiento de Modelos**
Se utilizan dos modelos vistos en clase:
- **Multinomial Naive Bayes**
- **Logistic Regression**

In [ ]:

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression

nb = MultinomialNB()
nb.fit(X_train, y_train)
pred_nb = nb.predict(X_test)

lr = LogisticRegression(max_iter=200)
lr.fit(X_train, y_train)
pred_lr = lr.predict(X_test)


## **5. Evaluación y Comparación con Modelo Pre‑entrenado (TextBlob)**

In [ ]:

from sklearn.metrics import classification_report
from textblob import TextBlob

print("### NAIVE BAYES ###")
print(classification_report(y_test, pred_nb))

print("### LOGISTIC REGRESSION ###")
print(classification_report(y_test, pred_lr))

def tb_sentiment(x):
    pol = TextBlob(x).sentiment.polarity
    if pol > 0: return 4
    if pol < 0: return 0
    return 2

test_df['tb_pred'] = test_df['text'].apply(tb_sentiment)

print("### TEXTBLOB ###")
print(classification_report(test_df['polarity'], test_df['tb_pred']))


## **Wordcloud (Opcional)**

In [ ]:

from wordcloud import WordCloud
import matplotlib.pyplot as plt

wc = WordCloud(width=900, height=450).generate(' '.join(train_df['clean'][:50000]))

plt.figure(figsize=(12,6))
plt.imshow(wc)
plt.axis('off')
plt.show()
